In [ ]:
#Importing necessary libraries
import pandas as pd
import numpy as np

In [ ]:
#Importing all csv files to dataframes

tayara_path = r'data_wrangling\csv\tayara_tn_standardized.csv'
automobile_path = r'data_wrangling\csv\automobiletn_set.csv'
wayback_path = r'data_wrangling\csv\tunisie_annonce_standardized.csv'

ty = pd.read_csv(tayara_path)
am = pd.read_csv(automobile_path)
wb = pd.read_csv(wayback_path)

In [ ]:
#To mitigate luxury cars' from automobile.tn data bias
luxury_brands = ['mercedes-benz','bmw','audi','land rover']
am_luxury = am[am['brand'].isin(luxury_brands)]
am_non_luxury = am[~am['brand'].isin(luxury_brands)]
am_luxury_sampled = am_luxury.sample(frac=0.42, random_state=42)
am = pd.concat([am_luxury_sampled, am_non_luxury])

In [ ]:

am.brand.value_counts()

In [ ]:
#Merging tayara and automobile first
ty = ty.drop(columns=['Unnamed: 0'])

In [ ]:
#Checking dtypes
ty.info()

In [ ]:
am.info()

In [ ]:
am['price'] = am['price'].astype('float64') #to fill them next, and because int64 does not support np.NaN
am['fiscal-power'] = am['fiscal-power'].astype('float64') #to fill them next
am = am.drop(columns=['title'])
am['circulation-date'] = pd.to_datetime(am['circulation-date'])
ty['circulation-date'] = pd.to_datetime(ty['circulation-date'])

In [ ]:
#Concatinating tayara and automobile dataframes
df = pd.concat([ty, am], ignore_index=True)

In [ ]:
#checking null values
null = df.isnull()
null.apply(pd.value_counts).fillna(0)

In [ ]:
#dealing with model typos using mapping
model_map = {
    'golf 4': 'golf', 'golf 5': 'golf', 'golf 6': 'golf', 'golf 7': 'golf',
    'golf 8': 'golf', 'golf plus': 'golf', 'volkswagen.golf 7': 'golf',
    'citroen.c3': 'c3',
    'fiat.punto': 'punto', 'grande punto': 'punto', 'punto evo': 'punto',
    'série 1': 'série 3', 'série 1 3p': 'série 3',
    'série 2 coupé': 'série 3', 'série 2 gran coupé': 'série 3',
    'série 2 active tourer': 'série 3',
    'série 3 coupé': 'série 3',
    'série 4 coupé': 'série 3', 'série 4 gran coupé': 'série 3',
    'série 5': 'série 3', 'série 6': 'série 3', 'série 6 gran coupé': 'série 3',
    'série 7': 'série 3',
    '206+': '206',
    '207': '207', '207 cc': '207',
    '208': '208', 'e-208': '208',
    '307': '307', '307 cc': '307',
    '308': '308',
    '3008 gt': '3008',
    '5008': '5008',
    '407': '407',
    '408 gt': '408',
    'partner': 'partner',
    'berlingo': 'berlingo', 'berlingo van': 'berlingo', 'berlingo utilitaire': 'berlingo',
    'megane cc': 'megane', 'megane sedan': 'megane',
    'corolla verso': 'corolla', 'corolla sedan': 'corolla',
    'yaris verso': 'yaris', 'yaris sedan': 'yaris',
    'wrangler unlimited': 'wrangler',
    'rio 5p': 'rio',
    '500 c': '500c',
    'classe a': 'a class',
    'classe c': 'c class',
    'classe e': 'e class'
}

df['model'] = df['model'].replace(model_map)


In [ ]:
#Imputing fiscal-power missing values:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# drop rows with missing fiscal-power
train_df = df[df['fiscal-power'].notna()]
test_df  = df[df['fiscal-power'].isna()]

# drop unusable columns (dates, price)
X = train_df.drop(columns=['fiscal-power', 'price', 'circulation-date', 'publish-date'])
y = train_df['fiscal-power']

X_test = test_df.drop(columns=['fiscal-power', 'price', 'circulation-date', 'publish-date'])

# one-hot encode categoricals
X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)

# align columns
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

# fit model
model = RandomForestRegressor(random_state=42)
model.fit(X, y)

# predict and round to integers
preds = model.predict(X_test)
preds_int = np.round(preds).astype(int)

# fill back
df.loc[df['fiscal-power'].isna(), 'fiscal-power'] = preds_int

In [ ]:
#Imputing car-age missing values:
# separate rows
train_df = df[df['car-age'].notna()]
test_df  = df[df['car-age'].isna()]

# features (drop target + unusable date/price)
X = train_df.drop(columns=['car-age', 'price', 'circulation-date', 'publish-date'])
y = train_df['car-age']

X_test = test_df.drop(columns=['car-age', 'price', 'circulation-date', 'publish-date'])

# one-hot encode categoricals
X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)

# align columns
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

# fit model
model = RandomForestRegressor(random_state=42)
model.fit(X, y)

# predict and round to integers
preds = model.predict(X_test)
preds_int = np.round(preds).astype(int)

# fill back
df.loc[df['car-age'].isna(), 'car-age'] = preds_int

In [ ]:
#Filling price missing values
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

df_imputed = df.copy()

has_price = df_imputed['price'].notna()
missing_price = df_imputed['price'].isna()

feature_columns = ['brand', 'model', 'mileage', 'fuel', 'engine-size', 
                  'gear', 'fiscal-power', 'body-type', 'location', 'car-age']

X_complete = df_imputed[has_price][feature_columns].copy()
y_complete = df_imputed[has_price]['price'].copy()
X_missing = df_imputed[missing_price][feature_columns].copy()

label_encoders = {}
categorical_columns = ['brand', 'model', 'fuel', 'gear', 'body-type', 'location']

for col in categorical_columns:
    if col in feature_columns:
        le = LabelEncoder()
        all_values = pd.concat([X_complete[col], X_missing[col]]).astype(str)
        le.fit(all_values)
        X_complete[col] = le.transform(X_complete[col].astype(str))
        X_missing[col] = le.transform(X_missing[col].astype(str))
        label_encoders[col] = le

X_complete = X_complete.fillna(X_complete.median())
X_missing = X_missing.fillna(X_complete.median())

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_complete, y_complete)

predicted_prices = rf_model.predict(X_missing)
predicted_prices = np.round(predicted_prices, -3)

df_imputed.loc[missing_price, 'price'] = predicted_prices

df = df_imputed

In [ ]:
#converting dtypes and creating new year column
df['price'] = df['price'].astype('int64')
df['engine-size'] = df['engine-size'].astype('int64')
df['fiscal-power'] = df['fiscal-power'].astype('int64')
df['car-age'] = df['car-age'].astype('int64')
df['year'] = df['circulation-date'].dt.year

In [ ]:
#mapping fuel types
fuel_mapping = {
    'essence': 'essence',
    'diesel': 'diesel',
    'electrique': 'electrique',
    'hybride': 'hybride',
    'hybride diesel': 'hybride diesel',
    'hybride essence': 'hybride essence',
    'hybride rechargeable essence': 'hybride essence',
    'hybride léger essence': 'hybride essence',
    'hybride léger diesel': 'hybride diesel',
    'hybride rechargeable diesel': 'hybride diesel',
}
df['fuel'] = df['fuel'].map(fuel_mapping)

In [ ]:
#mapping bodytype values
body_type_mapping = {
    'compacte': 'compacte',
    'berline': 'berline',
    '4 x 4': '4 x 4',
    'cabriolet': 'cabriolet',
    'monospace': 'monospace',
    'utilitaire': 'utilitaire',
    'pick up': 'pick up',
    'suv': 'suv',
    'citadine': 'compacte',
    'coupé': 'berline',
    'autres': 'utilitaire'
}
df['body-type'] = df['body-type'].map(body_type_mapping)

In [ ]:
wb.shape[0]

In [ ]:
df.shape[0]

In [ ]:
origin_map = {
    # German
    'volkswagen': 'german',
    'mercedes-benz': 'german',
    'bmw': 'german',
    'audi': 'german',
    'porsche': 'german',
    'opel': 'german',
    'smart': 'german',
    'skoda': 'german',
    'cupra': 'german',
    'seat' : 'german',

    # French
    'peugeot': 'french',
    'renault': 'french',
    'citroën': 'french',
    'ds': 'french',

    # Italian
    'fiat': 'italian',
    'alfa romeo': 'italian',
    'lancia': 'italian',
    'iveco': 'italian',

    # British
    'land rover': 'british',
    'jaguar': 'british',
    'mini': 'british',
    'mg': 'british',
    'ac': 'british',

    # American
    'ford': 'american',
    'chevrolet': 'american',
    'jeep': 'american',
    'hummer': 'american',
    'dodge': 'american',
    'tesla': 'american',

    # Japanese
    'toyota': 'japanese',
    'nissan': 'japanese',
    'mazda': 'japanese',
    'suzuki': 'japanese',
    'honda': 'japanese',
    'mitsubishi': 'japanese',
    'lexus': 'japanese',
    'isuzu': 'japanese',
    'infiniti': 'japanese',

    # Korean
    'kia': 'korean',
    'hyundai': 'korean',
    'ssangyong': 'korean',

    # Chinese
    'chery': 'chinese',
    'geely': 'chinese',
    'dongfeng': 'chinese',
    'great wall': 'chinese',
    'byd': 'chinese',
    'baic yx': 'chinese',
    'haval': 'chinese',
    'dfsk': 'chinese',
    'changan': 'chinese',
    'foton': 'chinese',
    'baic': 'chinese',

    # Indian
    'mahindra': 'indian',
    'tata': 'indian',

    # Romanian
    'dacia': 'romanian',

    # Swedish
    'volvo': 'swedish',

    # Russian
    'lada': 'russian',

    # Tunisian
    'wallyscar': 'tunisian'
}

df['origin'] = df['brand'].map(origin_map).fillna('unknown')
df_2025 = df

In [ ]:
def correct_body_type(model):
    """
    Mapping function to correct the body-type anomaly.
    Returns 'citadine' for models that are citadines (A/B segment),
    'compacte' for compactes (C segment), and None for others not fitting the categories.
    This addresses the issue where citadine models are misclassified as 'compacte'.
    To use in pandas: df['body-type'] = df['model'].apply(correct_body_type).fillna(df['body-type'])
    """
    mapping = {
        'rio': 'citadine',
        'palio': 'citadine',
        'polo': 'citadine',
        'picanto': 'citadine',
        'c3': 'citadine',
        'c1': 'citadine',
        'grand i10': 'citadine',
        'mito': 'citadine',
        'clio': 'citadine',
        '206': 'citadine',
        'i20': 'citadine',
        'celerio': 'citadine',
        'sandero': 'citadine',
        'punto': 'citadine',
        '500': 'citadine',
        'ibiza': 'citadine',
        'qq': 'citadine',
        'micra': 'citadine',
        '106': 'citadine',
        'fiesta': 'citadine',
        'ka': 'citadine',
        'seicento': 'citadine',
        'aygo': 'citadine',
        'yaris': 'citadine',
        'cooper': 'citadine',
        '208': 'citadine',
        'ypsilon': 'citadine',
        'fabia': 'citadine',
        'panda': 'citadine',
        '2': 'citadine',
        'aveo': 'citadine',
        'symbol': 'citadine',
        'swift': 'citadine',
        'note': 'citadine',
        'one': 'citadine',
        '107': 'citadine',
        'corsa': 'citadine',
        '207': 'citadine',
        'twingo': 'citadine',
        'smart': 'citadine',
        'qq6': 'citadine',
        'ax': 'citadine',
        '500c': 'citadine',
        'a1 sportback': 'citadine',
        'ds3': 'citadine',
        'logan': 'citadine',
        'spark': 'citadine',
        'siena': 'citadine',
        'c-elysée': 'citadine',
        'fox': 'citadine',
        'gol': 'citadine',
        '301': 'citadine',
        'panda city cross': 'citadine',
        'baleno': 'citadine',
        'agya': 'citadine',
        '108': 'citadine',
        'jazz': 'citadine',
        'sonic': 'citadine',
        'golf': 'compacte',
        'corolla': 'compacte',
        'leon': 'compacte',
        'a class': 'compacte',
        'a3': 'compacte',
        'megane': 'compacte',
        'escort': 'compacte',
        'ceed': 'compacte',
        '3': 'compacte',
        '308': 'compacte',
        'focus': 'compacte',
        'leon sc': 'compacte',
        'c4 cactus': 'compacte',
        'c4': 'compacte',
        'veloster': 'compacte',
        'astra': 'compacte',
        '306': 'compacte',
        '307': 'compacte',
        'new beetle': 'compacte',
        '19': 'compacte',
        'scirocco': 'compacte',
        'auris': 'compacte',
        'octavia': 'compacte',
        'rapid': 'compacte',
        'a3 sportback': 'compacte',
        'giulietta': 'compacte',
        "cee'd": 'compacte',
        'i30': 'compacte',
        'cerato 5p': 'compacte',
        'cla': 'compacte',
        'classe b': 'compacte',
        'altea': 'compacte',
        'modus': 'compacte',  # Though B MPV, but close
        'meriva': 'compacte',  # B MPV
        'ds5': 'compacte',
        'clubman': 'compacte'
    }
    return mapping.get(model.lower(), None)

df_2025['body-type'] = df_2025['model'].apply(correct_body_type).fillna(df_2025['body-type'])

In [ ]:
#Engineering new body-type column for the Time Series dataset using GROK AI (mention in the report)

def add_body_type(df):
    """
    Add a body_type column to a DataFrame based on brand and model.
    
    Parameters:
    df (pandas.DataFrame): DataFrame with 'brand' and 'model' columns.
    
    Returns:
    pandas.DataFrame: DataFrame with a new 'body_type' column.
    """
    # Define the body type mapping dictionary with updated classifications
    body_type_mapping = {
        ('alfa romeo', '147'): 'compacte',
        ('alfa romeo', 'mito'): 'citadine',
        ('alfa romeo', 'giulietta'): 'compacte',
        ('alfa romeo', '156'): 'berline',
        ('audi', 'a6'): 'berline',
        ('audi', 'a4'): 'berline',
        ('audi', 'a3'): 'compacte',
        ('audi', '80'): 'berline',
        ('audi', 'a2'): 'citadine',
        ('audi', 'q5'): 'suv',
        ('audi', '100'): 'berline',
        ('audi', 'a5'): 'berline',
        ('audi', 'a1 sportback'): 'citadine',
        ('audi', 'q7'): 'suv',
        ('audi', 'q3'): 'suv',
        ('baic', 'baic kenbo'): 'suv',
        ('bmw', 'série 3'): 'berline',
        ('bmw', 'série 5'): 'berline',
        ('bmw', 'série 7'): 'berline',
        ('bmw', 'm5'): 'berline',
        ('bmw', 'z4'): 'cabriolet',
        ('bmw', 'série 1'): 'berline',
        ('bmw', 'x5'): 'suv',
        ('bmw', 'x3'): 'suv',
        ('bmw', 'm3'): 'berline',
        ('bmw', 'x1'): 'suv',
        ('bmw', 'x6'): 'suv',
        ('chery', 'qq'): 'citadine',
        ('chery', 'tiggo 8'): 'suv',
        ('chery', 'arrizo'): 'berline',
        ('chevrolet', 'tahoe'): 'suv',
        ('chevrolet', 'matiz'): 'citadine',
        ('chevrolet', 'alero'): 'berline',
        ('chevrolet', 'aveo'): 'citadine',
        ('chevrolet', 'optra'): 'berline',
        ('chevrolet', 'cruze'): 'berline',
        ('citroën', 'xantia'): 'berline',
        ('citroën', 'berlingo'): 'utilitaire',
        ('citroën', 'c5'): 'berline',
        ('citroën', 'saxo'): 'citadine',
        ('citroën', 'jumper'): 'utilitaire',
        ('citroën', 'c3'): 'citadine',
        ('citroën', 'zx'): 'compacte',
        ('citroën', 'c4'): 'compacte',
        ('citroën', 'c15'): 'utilitaire',
        ('citroën', '2cv'): 'citadine',
        ('citroën', 'jumpy combi'): 'monospace',
        ('citroën', 'c4 picasso'): 'monospace',
        ('citroën', 'bx'): 'berline',
        ('citroën', 'nemo'): 'utilitaire',
        ('citroën', 'mehari'): 'cabriolet',
        ('citroën', 'ds4'): 'compacte',
        ('citroën', 'c1'): 'citadine',
        ('citroën', 'ds5'): 'suv',
        ('citroën', 'ds3'): 'citadine',
        ('dacia', 'logan'): 'berline',
        ('dacia', 'duster'): 'suv',
        ('dacia', 'sandero'): 'citadine',
        ('daewoo', 'nubira'): 'berline',
        ('daewoo', 'matiz'): 'citadine',
        ('dongfeng', 's50'): 'berline',
        ('fiat', 'fiat.punto'): 'citadine',
        ('fiat', 'doblo'): 'utilitaire',
        ('fiat', 'panda'): 'citadine',
        ('fiat', 'stilo'): 'compacte',
        ('fiat', 'uno'): 'citadine',
        ('fiat', 'tipo 5 portes'): 'compacte',
        ('fiat', '500'): 'citadine',
        ('fiat', 'grande punto'): 'citadine',
        ('fiat', 'bravo'): 'compacte',
        ('fiat', 'ritmo'): 'compacte',
        ('fiat', 'brava'): 'compacte',
        ('fiat', 'fiorino'): 'utilitaire',
        ('fiat', 'ducato'): 'utilitaire',
        ('fiat', 'linea'): 'berline',
        ('ford', 'escort'): 'compacte',
        ('ford', 'fiesta'): 'citadine',
        ('ford', 'mondeo'): 'berline',
        ('ford', 'galaxy'): 'monospace',
        ('ford', 'focus'): 'compacte',
        ('ford', 'sierra'): 'berline',
        ('ford', 'explorer'): 'suv',
        ('ford', 'transit'): 'utilitaire',
        ('ford', 'ranger'): 'pick up',
        ('ford', 'mustang'): 'cabriolet',
        ('ford', 'c-max'): 'monospace',
        ('ford', 'ka'): 'citadine',
        ('ford', 'maverick'): 'pick up',
        ('ford', 'fusion'): 'berline',
        ('great wall', 'm4'): 'suv',
        ('honda', 'civic'): 'compacte',
        ('honda', 'prelude'): 'cabriolet',
        ('honda', 'hr-v'): 'suv',
        ('honda', 'accord'): 'berline',
        ('honda', 'crx'): 'cabriolet',
        ('hyundai', 'tucson'): 'suv',
        ('hyundai', 'accent'): 'citadine',
        ('hyundai', 'elantra'): 'berline',
        ('hyundai', 'h1'): 'monospace',
        ('hyundai', 'i30'): 'compacte',
        ('hyundai', 'atos prime'): 'citadine',
        ('infiniti', 'fx35'): 'suv',
        ('isuzu', 'd-max'): 'pick up',
        ('isuzu', 'trooper'): 'suv',
        ('jeep', 'grand cherokee'): 'suv',
        ('jeep', 'cherokee'): 'suv',
        ('kia', 'rio'): 'citadine',
        ('kia', 'picanto'): 'citadine',
        ('kia', 'ceed'): 'compacte',
        ('kia', 'cerato 5p'): 'compacte',
        ('kia', 'sorento'): 'suv',
        ('kia', 'sportage'): 'suv',
        ('kia', 'carens'): 'monospace',
        ('kia', 'k2700'): 'utilitaire',
        ('land rover', '75'): 'berline',
        ('land rover', 'discovery'): 'suv',
        ('land rover', 'defender'): '4 x 4',
        ('land rover', 'range rover'): 'suv',
        ('land rover', 'freelander'): 'suv',
        ('lexus', 'ls'): 'berline',
        ('mazda', '6'): 'berline',
        ('mazda', 'cx-7'): 'suv',
        ('mazda', 'b 2500'): 'pick up',
        ('mazda', 'bt-50'): 'pick up',
        ('mazda', '3'): 'compacte',
        ('mercedes-benz', 'classe e'): 'berline',
        ('mercedes-benz', 'classe a'): 'compacte',
        ('mercedes-benz', 'ml'): 'suv',
        ('mercedes-benz', 'sl'): 'cabriolet',
        ('mercedes-benz', '230'): 'berline',
        ('mercedes-benz', 'classe b'): 'monospace',
        ('mercedes-benz', '240sx'): 'cabriolet',
        ('mercedes-benz', '190'): 'berline',
        ('mercedes-benz', 'cls'): 'berline',
        ('mercedes-benz', 'clk'): 'cabriolet',
        ('mercedes-benz', 'sprinter van'): 'utilitaire',
        ('mercedes-benz', '280'): 'berline',
        ('mercedes-benz', '300'): 'berline',
        ('mercedes-benz', 'slk'): 'cabriolet',
        ('mercedes-benz', 'vito'): 'utilitaire',
        ('mercedes-benz', 'cl'): 'berline',
        ('mercedes-benz', 'gle'): 'suv',
        ('mercedes-benz', 'glc'): 'suv',
        ('mercedes-benz', 'glb'): 'suv',
        ('mercedes-benz', 'vaneo'): 'monospace',
        ('mg', 'zs'): 'suv',
        ('mini', 'cooper'): 'citadine',
        ('nissan', '100'): 'berline',
        ('opel', 'corsa'): 'citadine',
        ('opel', 'astra'): 'compacte',
        ('opel', 'zafira'): 'monospace',
        ('opel', 'vectra'): 'berline',
        ('opel', 'kadett'): 'compacte',
        ('opel', 'agila'): 'monospace',
        ('opel', 'meriva'): 'monospace',
        ('opel', 'frontera'): 'suv',
        ('peugeot', 'partner'): 'utilitaire',
        ('peugeot', '106'): 'citadine',
        ('peugeot', '205'): 'citadine',
        ('peugeot', '406'): 'berline',
        ('peugeot', '206'): 'citadine',
        ('peugeot', '607'): 'berline',
        ('peugeot', '307'): 'compacte',
        ('peugeot', '405'): 'berline',
        ('peugeot', '305'): 'berline',
        ('peugeot', '407'): 'berline',
        ('peugeot', '207'): 'citadine',
        ('peugeot', 'boxer'): 'utilitaire',
        ('peugeot', '605'): 'berline',
        ('peugeot', '309'): 'compacte',
        ('peugeot', '504'): 'berline',
        ('peugeot', 'expert'): 'utilitaire',
        ('peugeot', '308'): 'compacte',
        ('peugeot', '208'): 'citadine',
        ('peugeot', '404'): 'berline',
        ('peugeot', '3008'): 'suv',
        ('peugeot', '306'): 'compacte',
        ('peugeot', '1007'): 'monospace',
        ('peugeot', '508'): 'berline',
        ('peugeot', 'bipper'): 'utilitaire',
        ('peugeot', '304'): 'berline',
        ('peugeot', 'rcz'): 'cabriolet',
        ('peugeot', '4008'): 'suv',
        ('porsche', 'cayenne'): 'suv',
        ('renault', 'clio'): 'citadine',
        ('renault', 'scenic'): 'monospace',
        ('renault', 'master'): 'utilitaire',
        ('renault', 'kangoo'): 'utilitaire',
        ('renault', 'laguna'): 'berline',
        ('renault', 'megane'): 'compacte',
        ('renault', 'twingo'): 'citadine',
        ('renault', 'express'): 'utilitaire',
        ('renault', 'super 5'): 'citadine',
        ('renault', 'trafic'): 'utilitaire',
        ('renault', 'safrane'): 'berline',
        ('renault', 'symbol'): 'berline',
        ('renault', 'r19'): 'compacte',
        ('renault', 'fluence'): 'berline',
        ('renault', 'latitude'): 'berline',
        ('renault', 'r21'): 'berline',
        ('renault', 'r9'): 'compacte',
        ('renault', 'r4'): 'citadine',
        ('renault', 'r18'): 'berline',
        ('renault truck', 'b'): 'utilitaire',
        ('seat', 'ibiza'): 'citadine',
        ('seat', 'leon'): 'compacte',
        ('seat', 'arosa'): 'citadine',
        ('skoda', 'fabia'): 'citadine',
        ('skoda', 'octavia'): 'berline',
        ('smart', 'smart'): 'citadine',
        ('ssangyong', 'kyron'): 'suv',
        ('ssangyong', 'korando'): 'suv',
        ('ssangyong', 'actyon'): 'suv',
        ('subaru', 'impreza'): 'compacte',
        ('toyota', 'corolla'): 'compacte',
        ('toyota', 'celica'): 'cabriolet',
        ('toyota', 'yaris'): 'citadine',
        ('toyota', 'starlet'): 'citadine',
        ('toyota', 'rav 4'): 'suv',
        ('toyota', 'land cruiser'): 'suv',
        ('toyota', 'tercel'): 'citadine',
        ('toyota', 'previa'): 'monospace',
        ('volkswagen', 'golf'): 'compacte',
        ('volkswagen', 'polo'): 'citadine',
        ('volkswagen', 'passat'): 'berline',
        ('volkswagen', 'golf plus'): 'compacte',
        ('volkswagen', 'fox'): 'citadine',
        ('volkswagen', 'jetta'): 'berline',
        ('volkswagen', 'caddy'): 'utilitaire',
        ('volkswagen', 'new beetle'): 'compacte',
        ('volkswagen', 'touran'): 'monospace',
        ('volkswagen', 'eos'): 'cabriolet',
        ('volkswagen', 'touareg'): 'suv',
        ('volkswagen', 'caravelle'): 'monospace',
        ('volkswagen', 'bora'): 'berline',
        ('volkswagen', 'transporter'): 'utilitaire',
        ('volkswagen', 'scirocco'): 'cabriolet',
        ('volkswagen', 'vento'): 'berline',
        ('volkswagen', 'tiguan'): 'suv',
        ('volvo', '740'): 'berline',
        ('volvo', 's80'): 'berline',
        ('volvo', 'xc 90'): 'suv',
        ('volvo', 's60'): 'berline',
        ('volvo', 'v40'): 'compacte',
        ('volvo', '264'): 'berline',
        ('wallyscar', 'wallys'): 'suv'
    }
    
    # Create a copy of the input DataFrame to avoid modifying the original
    df = df.copy()
    
    # Standardize brand and model columns (lowercase to match mapping keys)
    df['brand'] = df['brand'].str.lower()
    df['model'] = df['model'].str.lower()
    
    # Add body_type column by mapping (brand, model) tuples
    df['body-type'] = df.apply(
        lambda row: body_type_mapping.get((row['brand'], row['model']), 'unknown'),
        axis=1
    )
    
    return df

# Example usage:
# df = pd.read_csv('chatgpt.csv')
# df_with_body_type = add_body_type(df)
# df_with_body_type.to_csv('chatgpt_updated.csv', index=True)

# Example usage:
df = wb
new_wb = add_body_type(df)

In [ ]:
#Move this part and the above part to the data wrangling step
def extract_date_info(df):
    df['publish-date'] = pd.to_datetime(df['publish-date'], format='%Y-%m-%d')
    df['month'] = df['publish-date'].dt.month
    df['quarter'] = df['publish-date'].dt.quarter
    return df

new_wb = extract_date_info(new_wb)
new_wb.sample(10)

In [ ]:
#saving the 2025 data to a dataframe
df_2025.to_csv('2025_data_v2.csv')

In [ ]:
new_wb.to_csv('historical_data.csv')
#don't forget to mention in the report the use of GROK AI